# BRFSS 2015 — ZIP preflight, Train/CV only
Raw input supplied by the user: `/content/drive/MyDrive/Diabetes_Statistics_2015/01_Raw_Data/LLCP2015XPT.zip`.

**Final Test stays locked.** Run All cannot evaluate it. `schema.reviewed.json` is a partial draft: official CDC access was blocked in the development environment, so every proposed code must still be checked. No BRFSS results are bundled. Do not set `codebook_reviewed=true` merely to pass a guard.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
from datetime import datetime, timezone
CHECKOUT=Path('/content/PPTK')
if not CHECKOUT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/khanhleviet2006/PPTK.git',str(CHECKOUT)],check=True)
PROJECT=CHECKOUT/'BRFSS2015_Improvement'
assert (PROJECT/'src/data_io.py').is_file(), 'Use the updated project checkout; existing files are never reset automatically.'
DATA=Path('/content/drive/MyDrive/Diabetes_Statistics_2015/01_Raw_Data/LLCP2015XPT.zip')
assert DATA.is_file(), f'ZIP not found: {DATA}'
RUNS=Path('/content/drive/MyDrive/BRFSS2015_Improvement_Runs')
RUNS.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
# Copy the partial schema into Drive only when it does not already exist.
# Review edits persist here across Colab disconnects; never overwrite them on Run All.
SCHEMA=RUNS/'schema.reviewed.json'
if not SCHEMA.exists():
    with SCHEMA.open('x',encoding='utf8') as f:
        f.write((PROJECT/'schema.reviewed.json').read_text(encoding='utf8'))
PREFLIGHT=RUNS/f'preflight_{STAMP}.json'
OUTPUT=RUNS/f'run_{STAMP}'
print('DATA:',DATA,'\nSCHEMA:',SCHEMA,'\nOUTPUT:',OUTPUT)
print('Existing schema preserved:',SCHEMA.is_file())


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(PROJECT/'requirements.txt')],check=True)
subprocess.run([sys.executable,'-m','unittest','discover','-s',str(PROJECT/'tests'),'-p','test_*.py'],check=True)
subprocess.run([sys.executable,str(PROJECT/'tests/smoke.py')],check=True)


## ZIP and XPT preflight — no fitting and no final evaluation
The reader accepts exactly one XPT member, refuses unsafe names/links/encryption/ambiguous archives, bounds decompressed size (4 GiB) and compression ratio (200), and reads to EOF to check CRC. It uses a disk-backed temporary file, removed on success or failure. It never extracts archive paths or writes the raw XPT to Drive. Ensure Colab has enough temporary disk for the full decompressed member and RAM for later training.

This step reports header metadata plus at most 1,000 sample records internally; it does not certify the entire raw domain. It performs no Train/Test split, preprocessing fit, model selection or scoring. A ZIP SHA256 establishes identity, not authenticity without a trusted reference hash.


In [ ]:
subprocess.run([sys.executable,str(PROJECT/'src/research.py'),'inspect','--data',str(DATA),'--schema',str(SCHEMA),'--output',str(PREFLIGHT)],check=True)
preflight=json.loads(PREFLIGHT.read_text())
print('XPT member:',preflight.get('member'))
print('Reported rows:',preflight['xpt_reported_rows'])
print('Missing required columns:',preflight['missing_required_columns'])
assert not preflight['missing_required_columns'], 'Resolve column/schema mismatch before training.'


## Full raw-code audit — no learning or model evaluation
This scans the whole XPT to count raw codes for the predefined candidates and target. It can use substantial RAM. Counts do not establish code meanings; compare them against the official codebook. No split, fitted preprocessing, feature selection or model scoring occurs. Original data and earlier audits are preserved.

In [ ]:
RAW_AUDIT=RUNS/f'raw_codes_{STAMP}'
subprocess.run([sys.executable,str(PROJECT/'src/research.py'),'audit-codes','--data',str(DATA),'--schema',str(SCHEMA),'--output',str(RAW_AUDIT)],check=True)
print('Full raw-code frequencies:',RAW_AUDIT/'raw_code_counts.csv')


## Official CDC source — retrieve evidence without approving the schema
The URLs below are intended CDC references; retrieval failed in the cloud development environment. This cell attempts retrieval from Colab using normal TLS verification. It stores a hash and access metadata for review, but **never changes review flags**. Read the official variable entries and calculated-variable definitions, record document version and actual page/table citations, and verify every missing/raw-domain/recode/scale decision in the Drive schema.


In [ ]:
import urllib.request, urllib.parse, hashlib
CODEBOOK_URL='https://www.cdc.gov/brfss/annual_data/2015/pdf/codebook15_llcp.pdf'
try:
    request=urllib.request.Request(CODEBOOK_URL,headers={'User-Agent':'BRFSS2015-Research/1.0'})
    with urllib.request.urlopen(request,timeout=60) as response:
        host=urllib.parse.urlparse(response.url).hostname or ''
        if host != 'cdc.gov' and not host.endswith('.cdc.gov'):
            raise ValueError('Unexpected non-CDC source redirect')
        pdf=response.read(25*1024**2+1)
        if len(pdf)>25*1024**2 or not pdf.startswith(b'%PDF-'):
            raise ValueError('Response is too large or is not a PDF; inspect the official annual landing page')
    sha=hashlib.sha256(pdf).hexdigest()
    pdf_path=RUNS/f'codebook15_llcp_{sha[:16]}.pdf'
    if not pdf_path.exists():
        with pdf_path.open('xb') as f:f.write(pdf)
    metadata={'requested_url':CODEBOOK_URL,'resolved_url':response.url,'sha256':sha,'bytes':len(pdf),'retrieved_at_utc':datetime.now(timezone.utc).isoformat(),'review_complete':False}
    metadata_path=RUNS/f'codebook_access_{STAMP}.json'
    with metadata_path.open('x') as f:json.dump(metadata,f,indent=2)
    print('Retrieved evidence for manual review:',pdf_path,metadata)
except Exception as exc:
    print('Codebook remains UNVERIFIED:',type(exc).__name__,str(exc))
    print('Official landing page: https://www.cdc.gov/brfss/annual_data/annual_2015.html')


## Review gate — resolve all items before Train
Check the raw file identity and prior split/Test-use history. Verify target DIABETE3, all predictor definitions/units/missing/valid codes, design columns and diagnosis-time ambiguity. `_AGEG5YR` is an inactive alternative, not years; never silently replace the age representation using Test scores. Composites remain disabled until defined and justified.

Edit `SCHEMA` on Drive only after review. Clear `unverified_items` only when each item is resolved, add actual citations and document hash, set each active feature's `verification_status="VERIFIED"` and `leakage_review="safe"`, then set `codebook_reviewed=true`. The codebook must not be approved on the basis of a successful file download or synthetic tests alone. Sample preflight is not a full-data audit; the Train loader checks coded domains before fitting and logs the audit.


In [ ]:
schema=json.loads(SCHEMA.read_text())
print('Outstanding items:',json.dumps(schema.get('unverified_items',[]),indent=2))
assert schema.get('codebook_reviewed') is True, 'STOP: official codebook review is incomplete.'
assert not schema.get('unverified_items'), 'STOP: unresolved requirements.'
assert all(v.get('verification_status')=='VERIFIED' and v.get('leakage_review')=='safe' for v in schema['features'].values()), 'STOP: active variables are not fully reviewed.'


## Train/CV — unchanged study protocol
Fixed seed 2026; first group-stratified holdout (~20%) or supplied valid split; 5-fold stratified group CV on Train only. All learned preprocessing/selection is fold-fitted. Same sensitivity-constrained selection and fixed 0.5 threshold; 60-hour soft sweep/tuning budget, 12 Optuna trials for each of at most two boosting families. The remaining 12 hours are reserved for audit/reporting. Colab disconnects can interrupt long runs; a suitable runtime and resources remain prerequisites.

Use a fresh output directory. Current shortlist has eight active original variables; top 10/15/20 therefore clamp to the available count, not distinct 20-variable experiments. No performance target or Chang-equivalence claim is assumed.


In [ ]:
subprocess.run([sys.executable,str(PROJECT/'src/research.py'),'train','--data',str(DATA),'--schema',str(SCHEMA),'--output',str(OUTPUT),'--folds','5','--hours','60','--trials','12'],check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(OUTPUT/'04_Model_Training/model_cv_comparison.csv'))
print((OUTPUT/'failures.json').read_text())
print((OUTPUT/'best_model_config.json').read_text())
print('Final Test remains LOCKED. No test scores were requested.')


## Final Test — LOCKED
There is deliberately no executable final-evaluation cell. The CLI also refuses `final` without explicit `--unlock-final`; the existing one-time evaluation marker and frozen-artifact hashes still apply. Unlocking belongs to a separate step after data/protocol review and final configuration selection. Do not use Test to choose features, models, thresholds or hypotheses. Comparison to Chang et al. (2022) requires verification of an equivalent cohort/label/evaluation protocol; exceeding a point benchmark alone is insufficient.
